# Estonian Early Childhood Screen Use, Parent-Child Verbal Interaction, and Language Assessment Survey Data Exploration with `mlcroissant`
This notebook provides a step-by-step template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.rm1q-q4zw/fair2.json`

### Important note
This dataset package appears to provide rich metadata, but the provided package summary indicates that the `recordSet` list is empty. Because all entities should be referenced by `@id`, this notebook first inspects the loaded metadata to confirm whether any record sets are actually available through `mlcroissant`. If none are present, the notebook still demonstrates a robust workflow for metadata exploration and includes conditional code for record loading if record sets are later added or exposed by the source.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant pandas matplotlib seaborn

## 1. Data Loading
Load metadata from the dataset using `mlcroissant`.

We keep the Croissant URL in a variable so it can be reused throughout the notebook.

In [ ]:
import json
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.rm1q-q4zw/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata as a single object
metadata = dataset.metadata

# Display basic dataset metadata safely without subscripting the metadata object
print('Dataset loaded successfully.')
print('Name:', getattr(metadata, 'name', 'N/A'))
print('Description:', getattr(metadata, 'description', 'N/A'))
print('Identifier:', getattr(metadata, 'identifier', 'N/A'))
print('Version:', getattr(metadata, 'version', 'N/A'))
print('License:', getattr(metadata, 'license', 'N/A'))

In [ ]:
# Convert metadata to a JSON-like structure for inspection when needed
# This avoids treating dataset.metadata itself as a dict.
metadata_json = metadata.to_json()

print(json.dumps({
    '@id': metadata_json.get('@id'),
    'name': metadata_json.get('name'),
    'identifier': metadata_json.get('identifier'),
    'version': metadata_json.get('version'),
    'temporalCoverage': metadata_json.get('temporalCoverage'),
    'spatialCoverage': metadata_json.get('spatialCoverage')
}, indent=2, ensure_ascii=False))

## 2. Data Overview
Review available record sets, fields, and their `@id` values.

Because this dataset package may contain metadata without exposed record sets, we inspect the metadata JSON representation to discover available entities.

In [ ]:
# Inspect top-level keys available in the metadata JSON
top_level_keys = sorted(metadata_json.keys())
print('Top-level metadata keys:')
print(top_level_keys)

In [ ]:
# Summarize important top-level metadata fields
summary_fields = {
    '@id': metadata_json.get('@id'),
    '@type': metadata_json.get('@type'),
    'name': metadata_json.get('name'),
    'identifier': metadata_json.get('identifier'),
    'datePublished': metadata_json.get('datePublished'),
    'temporalCoverage': metadata_json.get('temporalCoverage'),
    'spatialCoverage': metadata_json.get('spatialCoverage')
}

pd.DataFrame(summary_fields.items(), columns=['field', 'value'])

In [ ]:
# Inspect record sets using their @id values whenever available
record_sets = metadata_json.get('recordSet', []) or []

print(f'Number of record sets found: {len(record_sets)}')

record_set_rows = []
for rs in record_sets:
    record_set_rows.append({
        'record_set_@id': rs.get('@id'),
        'name': rs.get('name'),
        'description': rs.get('description'),
        'field_count': len(rs.get('field', []) or [])
    })

record_sets_df = pd.DataFrame(record_set_rows)
record_sets_df

If record sets are available, the next cell lists the fields in each record set using their `@id` values. If no record sets are available, the result will simply be empty.

In [ ]:
field_rows = []

for rs in record_sets:
    rs_id = rs.get('@id')
    for field in rs.get('field', []) or []:
        source_info = field.get('source', {}) if isinstance(field.get('source', {}), dict) else {}
        field_rows.append({
            'record_set_@id': rs_id,
            'field_@id': field.get('@id'),
            'field_name': field.get('name'),
            'description': field.get('description'),
            'dataType': field.get('dataType'),
            'source_fileObject_@id': source_info.get('fileObject'),
            'source_extract_column': (source_info.get('extract') or {}).get('column') if isinstance(source_info.get('extract'), dict) else None
        })

fields_df = pd.DataFrame(field_rows)
fields_df.head(20)

The dataset package also includes rich descriptive metadata that can be explored even when tabular record sets are absent.

In [ ]:
# Explore selected metadata lists and scalar values
descriptive_rows = [
    {'metadata_field': 'keywords', 'value': metadata_json.get('keywords')},
    {'metadata_field': 'dataCollectionType', 'value': metadata_json.get('dataCollectionType')},
    {'metadata_field': 'dataUseCases', 'value': metadata_json.get('dataUseCases')},
    {'metadata_field': 'dataLimitations', 'value': metadata_json.get('dataLimitations')},
    {'metadata_field': 'dataBiases', 'value': metadata_json.get('dataBiases')},
    {'metadata_field': 'personalSensitiveInformation', 'value': metadata_json.get('personalSensitiveInformation')}
]

pd.DataFrame(descriptive_rows)

## 3. Data Extraction
Load data from available record sets into pandas DataFrames for analysis.

All record sets are referenced by their `@id`. Because this dataset currently appears to expose no record sets, the code is written defensively and will only attempt extraction when record sets are present.

In [ ]:
# Extract data from each record set using record set @id values
record_set_ids = [rs.get('@id') for rs in record_sets if rs.get('@id')]
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f'Loaded {len(records)} records from record set: {record_set_id}')
    except Exception as e:
        print(f'Could not load records for record set {record_set_id}: {e}')

print('Available extracted DataFrames:', list(dataframes.keys()))

In [ ]:
# Preview the first available DataFrame, if any
if dataframes:
    first_record_set_id = next(iter(dataframes))
    print('First record set @id:', first_record_set_id)
    print('Columns:')
    print(dataframes[first_record_set_id].columns.tolist())
    display(dataframes[first_record_set_id].head())
else:
    print('No record sets were extracted. This is consistent with the metadata summary showing an empty recordSet list.')

For completeness, the next cell shows how to inspect individual records by `record_set` `@id` when a record set exists.

In [ ]:
if record_set_ids:
    example_record_set_id = record_set_ids[0]
    print('Inspecting records from record set @id:', example_record_set_id)
    try:
        for i, record in enumerate(dataset.records(record_set=example_record_set_id)):
            print(record)
            if i >= 2:
                break
    except Exception as e:
        print('Unable to iterate through records:', e)
else:
    print('No record set @id is available for record-level inspection.')

## 4. Exploratory Data Analysis (EDA)
This section demonstrates common data processing patterns.

Because no tabular record set is currently exposed, we perform EDA on structured metadata and include conditional examples for record-level EDA if tabular data becomes available later.

In [ ]:
# Build a compact metadata analysis table
metadata_analysis = pd.DataFrame([
    {'metric_@id': 'keywords_count', 'value': len(metadata_json.get('keywords', []) or [])},
    {'metric_@id': 'authors_count', 'value': len(metadata_json.get('author', []) or [])},
    {'metric_@id': 'citations_count', 'value': len(metadata_json.get('citation', []) or [])},
    {'metric_@id': 'funding_count', 'value': len(metadata_json.get('funding', []) or [])},
    {'metric_@id': 'distributions_count', 'value': len(metadata_json.get('distribution', []) or [])},
    {'metric_@id': 'record_sets_count', 'value': len(metadata_json.get('recordSet', []) or [])},
    {'metric_@id': 'visualizations_count', 'value': len(metadata_json.get('visualization', []) or [])},
    {'metric_@id': 'data_biases_count', 'value': len(metadata_json.get('dataBiases', []) or [])},
    {'metric_@id': 'data_limitations_count', 'value': len(metadata_json.get('dataLimitations', []) or [])},
    {'metric_@id': 'use_cases_count', 'value': len(metadata_json.get('dataUseCases', []) or [])}
])

metadata_analysis

In [ ]:
# Example of filtering metadata-derived metrics
numeric_field_id = 'value'
threshold = 1

filtered_metrics_df = metadata_analysis[metadata_analysis[numeric_field_id] > threshold].copy()
print(f'Filtered metadata-derived metrics with {numeric_field_id} > {threshold}:')
display(filtered_metrics_df)

if not filtered_metrics_df.empty and filtered_metrics_df[numeric_field_id].std() not in [0, None]:
    filtered_metrics_df[f'{numeric_field_id}_normalized'] = (
        (filtered_metrics_df[numeric_field_id] - filtered_metrics_df[numeric_field_id].mean())
        / filtered_metrics_df[numeric_field_id].std()
    )
else:
    filtered_metrics_df[f'{numeric_field_id}_normalized'] = 0.0

print(f'Normalized {numeric_field_id} for filtered metadata metrics:')
display(filtered_metrics_df[[numeric_field_id, f'{numeric_field_id}_normalized']])

In [ ]:
# Optional record-level EDA template using @id references, if a record set becomes available
if dataframes:
    active_record_set_id = next(iter(dataframes))
    active_df = dataframes[active_record_set_id].copy()
    print('Running conditional EDA on record set @id:', active_record_set_id)
    
    numeric_columns = active_df.select_dtypes(include='number').columns.tolist()
    print('Numeric columns:', numeric_columns)
    
    if numeric_columns:
        numeric_field_id = numeric_columns[0]
        q1 = active_df[numeric_field_id].quantile(0.25)
        q3 = active_df[numeric_field_id].quantile(0.75)
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
        cleaned_df = active_df[
            (active_df[numeric_field_id] >= lower_bound) &
            (active_df[numeric_field_id] <= upper_bound)
        ].copy()
        
        if cleaned_df[numeric_field_id].std() not in [0, None]:
            cleaned_df[f'{numeric_field_id}_normalized'] = (
                (cleaned_df[numeric_field_id] - cleaned_df[numeric_field_id].mean()) /
                cleaned_df[numeric_field_id].std()
            )
        else:
            cleaned_df[f'{numeric_field_id}_normalized'] = 0.0
        
        print(f'Example numeric field @id/column selected: {numeric_field_id}')
        print('Cleaned DataFrame preview:')
        display(cleaned_df.head())
        
        group_candidates = [col for col in active_df.columns if active_df[col].dtype == 'object']
        if group_candidates:
            group_field_id = group_candidates[0]
            grouped_df = cleaned_df.groupby(group_field_id)[numeric_field_id].mean(numeric_only=False).reset_index()
            print(f'Grouped by field @id/column: {group_field_id}')
            display(grouped_df.head())
    else:
        print('No numeric columns found for record-level EDA.')
else:
    print('Skipping record-level EDA because no record sets were extracted.')

## 5. Visualization
Visualize metadata distributions and, conditionally, any tabular record data if present.

In [ ]:
# Visualization 1: metadata-derived counts
plt.figure(figsize=(10, 5))
sns.barplot(data=metadata_analysis.sort_values('value', ascending=False), x='metric_@id', y='value', palette='viridis')
plt.xticks(rotation=45, ha='right')
plt.title('Metadata-Derived Dataset Counts')
plt.xlabel('Metric @id')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Visualization 2: keyword table for quick inspection
keywords = metadata_json.get('keywords', []) or []
keywords_df = pd.DataFrame({'keyword_@id_or_value': keywords})
display(keywords_df.head(20))

print(f'Total keywords: {len(keywords_df)}')

In [ ]:
# Conditional visualization for the first available numeric column in the first available record set
if dataframes:
    active_record_set_id = next(iter(dataframes))
    active_df = dataframes[active_record_set_id]
    numeric_columns = active_df.select_dtypes(include='number').columns.tolist()
    
    if numeric_columns:
        numeric_field_id = numeric_columns[0]
        plt.figure(figsize=(8, 5))
        sns.histplot(active_df[numeric_field_id].dropna(), kde=True)
        plt.title(f'Distribution of {numeric_field_id} in {active_record_set_id}')
        plt.xlabel(numeric_field_id)
        plt.tight_layout()
        plt.show()
    else:
        print('No numeric fields available for record-level visualization.')
else:
    print('No record-level visualization generated because no record sets are available.')

## 6. Conclusion
This notebook demonstrated how to load and inspect a Croissant dataset with `mlcroissant`, while consistently referencing dataset entities by their `@id` values whenever available.

### Key observations
- The dataset metadata loads successfully from the Croissant URL.
- The package contains rich descriptive metadata about study design, limitations, biases, use cases, and governance.
- Based on the available metadata representation, the `recordSet` list is currently empty, so no tabular records were extracted in this run.
- The notebook includes reusable conditional code for record-set loading, field inspection, preprocessing, grouping, and visualization if record sets are later exposed.

### Next steps
- Re-run the notebook if the Croissant source is updated with record sets and fields.
- If a separate distribution file contains the tabular data, inspect the distribution `@id` and linked resources to determine whether additional access steps are required.
- Extend the conditional EDA section once field-level `@id` values become available.